In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import balanced_accuracy_score

In [2]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print(train.shape)
print(test.shape)

(577347, 12)
(247435, 11)


In [3]:
X = train.drop(columns=["id", "class"])
y = train["class"]

X_test = test.drop(columns=["id"])

# Color indices
X["u_g"] = X["u"] - X["g"]
X["g_r"] = X["g"] - X["r"]
X["r_i"] = X["r"] - X["i"]
X["i_z"] = X["i"] - X["z"]

X["u_r"] = X["u"] - X["r"]
X["g_i"] = X["g"] - X["i"]
X["r_z"] = X["r"] - X["z"]

X_test["u_g"] = X_test["u"] - X_test["g"]
X_test["g_r"] = X_test["g"] - X_test["r"]
X_test["r_i"] = X_test["r"] - X_test["i"]
X_test["i_z"] = X_test["i"] - X_test["z"]

X_test["u_r"] = X_test["u"] - X_test["r"]
X_test["g_i"] = X_test["g"] - X_test["i"]
X_test["r_z"] = X_test["r"] - X_test["z"]

In [4]:
X = X.drop(
    columns=["spectral_type", "galaxy_population"]
)

X_test = X_test.drop(
    columns=["spectral_type", "galaxy_population"]
)

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=8,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_valid)

    score = balanced_accuracy_score(
        y_valid,
        preds
    )

    scores.append(score)

    print(f"Fold Score: {score:.6f}")

print("\nMean CV Score:", np.mean(scores))

Fold Score: 0.953735
Fold Score: 0.956107
Fold Score: 0.953874
Fold Score: 0.954148
Fold Score: 0.953801

Mean CV Score: 0.9543328700623788


In [10]:
model_lgbm_v1 = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    verbosity=-1
)

model_lgbm_v1.fit(X, y)

,max_depth,8
,learning_rate,0.05
,n_estimators,1000
,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [12]:
import joblib

joblib.dump(
    model_lgbm_v1,
    "../models/lightgbm_v1.pkl"
)

['../models/lightgbm_v1.pkl']

In [13]:
preds = model_lgbm_v1.predict(X_test)

submission = pd.DataFrame({
    "id": test["id"],
    "class": preds
})

submission.to_csv(
    "../submissions/submission_v6.csv",
    index=False
)

In [5]:
import joblib

model_lgbm_v1 = joblib.load("../models/lightgbm_v1.pkl")

### Experiment V7: LightGBM Feature Importance

In [8]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model_lgbm_v1.feature_importances_
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
1,delta,12113
0,alpha,12011
7,redshift,11059
14,r_z,5386
13,g_i,5097
2,u,4899
6,z,4888
8,u_g,4621
12,u_r,4610
9,g_r,4546


### Instead of feature removal, let's test whether LightGBM can benefit from more expressive trees

In [9]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = LGBMClassifier(
        n_estimators=1500,
        learning_rate=0.03,
        max_depth=8,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_valid)

    score = balanced_accuracy_score(
        y_valid,
        preds
    )

    scores.append(score)

    print(f"Fold Score: {score:.6f}")

print("\nMean CV Score:", np.mean(scores))

Fold Score: 0.954357
Fold Score: 0.956268
Fold Score: 0.955372
Fold Score: 0.955144
Fold Score: 0.955434

Mean CV Score: 0.9553151504650105


In [11]:
model_lgbm_v2 = LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=31,
    random_state=42,
    verbosity=-1
)

model_lgbm_v2.fit(X, y)

,max_depth,8
,learning_rate,0.03
,n_estimators,1500
,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [12]:
import joblib
joblib.dump(model_lgbm_v2, "../models/lightgbm_v2.pkl")

['../models/lightgbm_v2.pkl']

In [13]:
preds = model_lgbm_v2.predict(X_test)

submission = pd.DataFrame({
    "id": test["id"],
    "class": preds
})

submission.to_csv(
    "../submissions/submission_v7.csv",
    index=False
)

In [14]:
import joblib

model_lgbm_v2 = joblib.load("../models/lightgbm_v2.pkl")

### Advanced Feature Engineering

In [6]:
# Ratios
X["u_over_g"] = X["u"] / (X["g"] + 1e-6)
X["g_over_r"] = X["g"] / (X["r"] + 1e-6)
X["r_over_i"] = X["r"] / (X["i"] + 1e-6)
X["i_over_z"] = X["i"] / (X["z"] + 1e-6)

X_test["u_over_g"] = X_test["u"] / (X_test["g"] + 1e-6)
X_test["g_over_r"] = X_test["g"] / (X_test["r"] + 1e-6)
X_test["r_over_i"] = X_test["r"] / (X_test["i"] + 1e-6)
X_test["i_over_z"] = X_test["i"] / (X_test["z"] + 1e-6)

In [8]:
X["redshift_u"] = X["redshift"] * X["u"]
X["redshift_g"] = X["redshift"] * X["g"]
X["redshift_r"] = X["redshift"] * X["r"]

X_test["redshift_u"] = X_test["redshift"] * X_test["u"]
X_test["redshift_g"] = X_test["redshift"] * X_test["g"]
X_test["redshift_r"] = X_test["redshift"] * X_test["r"]

In [9]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=8,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_valid)

    score = balanced_accuracy_score(
        y_valid,
        preds
    )

    scores.append(score)

    print(f"Fold Score: {score:.6f}")

print("\nMean CV Score:", np.mean(scores))

Fold Score: 0.953113
Fold Score: 0.955232
Fold Score: 0.954170
Fold Score: 0.954970
Fold Score: 0.955463

Mean CV Score: 0.9545895547611888


In [10]:
model_lgbm_v3 = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    verbosity=-1
)

model_lgbm_v3.fit(X, y)

,max_depth,8
,learning_rate,0.05
,n_estimators,1000
,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [11]:
import joblib

joblib.dump(
    model_lgbm_v3,
    "../models/lightgbm_v3.pkl"
)

['../models/lightgbm_v3.pkl']

In [12]:
preds = model_lgbm_v3.predict(X_test)

submission = pd.DataFrame({
    "id": test["id"],
    "class": preds
})

submission.to_csv(
    "../submissions/submission_v10.csv",
    index=False
)

In [ ]:
import joblib

model_lgbm_v1 = joblib.load("../models/lightgbm_v1.pkl")